In [1]:
import pandas as pd
from pathlib import Path

In [7]:
uy_attributes_dir = Path("/home/azureuser/NeuralHydrologyAzure/1_preparing_data/1_uy_basins_data/data/attributes/attributes.csv")
us_attributes_dir_1 = Path("/home/azureuser/NeuralHydrologyAzure/1_preparing_data/2_us_127_basins/data/attributes/attributes_caravan.csv")
us_attributes_dir_2 = Path("/home/azureuser/NeuralHydrologyAzure/1_preparing_data/2_us_127_basins/data/attributes/attributes_hydroatlas.csv")
us_attributes_dir_3 = Path("/home/azureuser/NeuralHydrologyAzure/1_preparing_data/2_us_127_basins/data/attributes/attributes_other.csv")
us_basins_dir = Path("/home/azureuser/NeuralHydrologyAzure/2_final_runs/2_us_127_basins/1_basins_subset_test.txt")

In [6]:
uy_attributes = pd.read_csv(uy_attributes_dir)
uy_attributes.describe()

,slope_mean,high_prec_freq,p_mean,high_prec_dur,low_prec_freq,silt_frac,aridity,pet_mean,frac_snow,clay_frac,area_gages2,sand_frac,elev_mean,low_prec_dur
count,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.0,16.000000,16.000000,16.000000,16.000000,16.000000
mean,19.728438,0.064605,3.781638,1.224120,0.691911,23.223874,0.759631,2.863671,0.0,28.877584,2877.830045,47.957333,163.434616,4.677805
std,8.180745,0.001076,0.233502,0.010773,0.019419,1.582358,0.049056,0.101881,0.0,2.457892,2873.523409,3.427123,45.448346,0.203697
min,11.890806,0.062628,3.325953,1.203846,0.664613,21.110253,0.712604,2.687766,0.0,25.547095,314.411261,41.744305,71.728083,4.354260
25%,13.949711,0.063895,3.661358,1.216984,0.678747,21.985117,0.724184,2.799710,0.0,26.917009,957.602135,46.129602,150.648296,4.568376
50%,16.265903,0.064579,3.892910,1.224527,0.684736,23.291298,0.741529,2.880539,0.0,29.381229,1652.165947,47.338232,173.009974,4.609556
75%,24.890294,0.065418,3.936994,1.231739,0.706656,24.020669,0.793075,2.946373,0.0,30.289850,3447.207916,51.400092,198.245915,4.792610
max,37.254440,0.066256,4.062473,1.243871,0.730322,26.581849,0.884248,2.993270,0.0,34.177618,8881.479722,52.807840,214.944342,5.027247


In [13]:
us_basins = pd.read_csv(us_basins_dir, header=None)
us_basins

,0
0,camels_01411300
1,camels_01466500
2,camels_01487000
3,camels_01638480
4,camels_01644000
...,...
122,camels_09505800
123,camels_10259000
124,camels_11451100
125,camels_12073500


In [19]:
basin_ids = us_basins[0].tolist()
len(basin_ids)

127

In [22]:
us_attributes_1 = pd.read_csv(us_attributes_dir_1)
us_attributes_1 = us_attributes_1[us_attributes_1["gauge_id"].isin(basin_ids)]

us_attributes_2 = pd.read_csv(us_attributes_dir_2)
us_attributes_2 = us_attributes_2[us_attributes_2["gauge_id"].isin(basin_ids)]

us_attributes_3 = pd.read_csv(us_attributes_dir_3)
us_attributes_3 = us_attributes_3[us_attributes_3["gauge_id"].isin(basin_ids)]

In [24]:
rename_map = {
    "slp_dg_sav": "slope_mean",
    "p_mean": "p_mean",
    "high_prec_dur": "high_prec_dur",
    "low_prec_dur": "low_prec_dur",
    "slt_pc_sav": "silt_frac",
    "aridity_FAO_PM": "aridity",
    "pet_mean_FAO_PM": "pet_mean",
    "cly_pc_sav": "clay_frac",
    "area": "area_gages2",
    "snd_pc_sav": "sand_frac",
    "ele_mt_sav": "elev_mean",
}

csv_cols = list(rename_map.keys())
basin_ids = us_basins[0].str.strip().tolist()

dfs = []
for path in [us_attributes_dir_1, us_attributes_dir_2, us_attributes_dir_3]:
    df = pd.read_csv(path)
    df = df[df["gauge_id"].isin(basin_ids)]
    keep = ["gauge_id"] + [c for c in csv_cols if c in df.columns]
    dfs.append(df[keep].set_index("gauge_id"))

attributes = pd.concat(dfs, axis=1)
attributes = attributes.loc[:, ~attributes.columns.duplicated()]
attributes = attributes[csv_cols].rename(columns=rename_map).reset_index()

# original column order (only those that exist)
original_order = ["gauge_id", "slope_mean", "high_prec_freq", "p_mean", "high_prec_dur",
                  "low_prec_freq", "silt_frac", "aridity", "pet_mean", "frac_snow",
                  "clay_frac", "area_gages2", "sand_frac", "elev_mean", "low_prec_dur"]
attributes = attributes[[c for c in original_order if c in attributes.columns]]

attributes.head()

,gauge_id,slope_mean,p_mean,high_prec_dur,silt_frac,aridity,pet_mean,clay_frac,area_gages2,sand_frac,elev_mean,low_prec_dur
0,camels_01411300,3.114887,3.046962,1.119197,31.033039,0.850316,2.590882,13.971485,79.230476,55.937773,20.118898,3.687100
1,camels_01466500,5.000000,3.193320,1.107417,25.213958,0.800156,2.555153,5.213958,6.303866,69.572085,39.641873,3.537472
2,camels_01487000,3.961830,3.132572,1.140236,33.961830,0.816437,2.557548,18.000000,187.283875,48.038170,16.000000,3.677091
3,camels_01638480,32.000000,2.883632,1.155405,41.000000,0.788938,2.275008,18.000000,233.587289,41.000000,170.000000,3.756209
4,camels_01644000,37.277196,2.882359,1.146575,41.088645,0.786511,2.267007,18.544308,858.811596,40.653099,188.050313,3.723380


In [25]:
attributes.describe()

,slope_mean,p_mean,high_prec_dur,silt_frac,aridity,pet_mean,clay_frac,area_gages2,sand_frac,elev_mean,low_prec_dur
count,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000
mean,31.837889,3.064291,1.211568,35.907220,1.043280,2.939170,21.196851,589.290949,42.915772,289.339418,4.492567
std,36.144338,0.594641,0.070245,4.942739,0.485524,0.435840,4.546013,488.225924,8.633015,304.012962,1.264821
min,0.320110,0.731233,1.107417,21.849690,0.550236,1.898078,5.213958,6.303866,21.258818,16.000000,3.013498
25%,9.925748,2.906156,1.167265,33.519112,0.797570,2.576190,18.892018,209.866196,37.860234,88.528397,3.860667
50%,17.406489,3.184932,1.186691,36.037986,0.853415,2.977704,21.000000,451.681722,43.000000,188.050313,4.121080
75%,36.982254,3.453118,1.237044,38.073186,1.072900,3.199195,24.000000,855.567693,47.000000,379.997365,4.864599
max,170.092340,4.170782,1.496875,51.580394,4.145547,3.994414,33.877580,1929.072084,69.572085,2021.082672,13.356037


In [26]:
# columns present in both (excluding the id)
common = [c for c in attributes.columns
          if c in uy_attributes.columns and c != "gauge_id"]

comparison = pd.DataFrame({
    "US_mean": attributes[common].mean(),
    "UY_mean": uy_attributes[common].mean(),
})
comparison["diff (UY - US)"] = comparison["UY_mean"] - comparison["US_mean"]
comparison["diff_%"] = 100 * comparison["diff (UY - US)"] / comparison["US_mean"]

comparison.round(3)

,US_mean,UY_mean,diff (UY - US),diff_%
slope_mean,31.838,19.728,-12.109,-38.035
p_mean,3.064,3.782,0.717,23.410
high_prec_dur,1.212,1.224,0.013,1.036
silt_frac,35.907,23.224,-12.683,-35.323
aridity,1.043,0.760,-0.284,-27.188
pet_mean,2.939,2.864,-0.075,-2.569
clay_frac,21.197,28.878,7.681,36.235
area_gages2,589.291,2877.830,2288.539,388.355
sand_frac,42.916,47.957,5.042,11.748
elev_mean,289.339,163.435,-125.905,-43.515
